In [1]:
# install libraries if not already installed
# pip install pandas pandas_datareader fredapi matplotlib seaborn numpy python-dotenv

import pandas as pd
import pandas_datareader as pdr
from fredapi import Fred
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv


In [2]:

# Load environment variables
load_dotenv()

# Configure FRED API
fred_key = os.getenv('FRED_API_KEY')
if not fred_key:
    raise ValueError("FRED_API_KEY not found in environment variables.")
fred = Fred(api_key=fred_key)

# Create output directory
output_dir = 'data_csv'
os.makedirs(output_dir, exist_ok=True)

In [3]:
# --- Date Configuration ---

# 1. VAR Analysis Data
# Timeframe: 1987 to mid-2007 (pre-crisis)
VAR_START_DATE = '1987-01-01'
VAR_END_DATE = '2007-07-01'

# 2. Visuals & Analysis Data
# Timeframe: 2000 to present (for broader context)
VISUALS_START_DATE = '2000-01-01'
VISUALS_END_DATE = '2025-08-01'

# 3. Specific Series Overrides
HOUSING_STARTS_START = '1959-01-01' # Long-term historical context

In [4]:
# --- Ticker Definitions ---

# National Level Data
national_tickers = {
    # VAR Data Series
    'cs': 'CSUSHPINSA',       # Case-Shiller National Home Price Index
    'cs10': 'SPCS10RSA',      # Case-Shiller 10-City Home Price Index
    'pce': 'PCEPI',           # Consumer Price Index (PCE)
    'ffr': 'FEDFUNDS',        # Federal Funds Rate
    'unemp': 'UNRATE',        # Unemployment Rate
    'pce_rate': 'PCETRIM12M159SFRBDAL', # PCE 12-Month Rate of Change
    
    # Visuals Data Series
    'permits': 'PERMIT',      # Building Permits
    'population': 'POPTHM',   # Population (thousands)
    'housing starts': 'HOUST',# Housing Starts
    'recessions': 'USREC',    # Recession Indicator
}

In [5]:
# Dictionary of tickers for the top 20 Metropolitan Statistical Areas (MSAs)
# HPI: S&P/Case-Shiller Home Price Index (Seasonally Adjusted)
# Permits: New Private Housing Units Authorized by Building Permits (Seasonally Adjusted)
# Population: Resident Population (Annual, interpolated to Quarterly if needed)

metropol_tickers = {
    'Atlanta': {'hpi': 'ATXRSA', 'permits': 'ATLA013BPPRIVSA', 'pop': 'ATLPOP'},
    'Boston': {'hpi': 'BOXRSA', 'permits': 'BOST625BPPRIVSA', 'pop': 'BOSPOP'},
    'Charlotte': {'hpi': 'CRXRSA', 'permits': 'CHAR737BPPRIVSA', 'pop': 'CGRPOP'},
    'Chicago': {'hpi': 'CHXRSA', 'permits': 'CHIC917BPPRIVSA', 'pop': 'CHIPOP'},
    'Cleveland': {'hpi': 'CEXRSA', 'permits': 'CLEV439BPPRIVSA', 'pop': 'CVLPOP'},
    'Dallas': {'hpi': 'DAXRSA', 'permits': 'DALL148BPPRIVSA', 'pop': 'DFWPOP'},
    'Denver': {'hpi': 'DNXRSA', 'permits': 'DENV708BPPRIVSA', 'pop': 'DNVPOP'},
    'Detroit': {'hpi': 'DEXRSA', 'permits': 'DETR826BPPRIVSA', 'pop': 'DWLPOP'},
    'Las Vegas': {'hpi': 'LVXRSA', 'permits': 'LASV832BPPRIVSA', 'pop': 'LSVPOP'},
    'Los Angeles': {'hpi': 'LXXRSA', 'permits': 'LOSA106BPPRIVSA', 'pop': 'LNAPOP'},
    'Miami': {'hpi': 'MIXRSA', 'permits': 'MIAM112BPPRIVSA', 'pop': 'MIMPOP'},
    'Minneapolis': {'hpi': 'MNXRSA', 'permits': 'MINN427BPPRIVSA', 'pop': 'MSPPOP'},
    'New York': {'hpi': 'NYXRSA', 'permits': 'NEWY636BPPRIVSA', 'pop': 'NYTPOP'},
    'Phoenix': {'hpi': 'PHXRSA', 'permits': 'PHOE004BPPRIVSA', 'pop': 'PHXPOP'},
    'Portland': {'hpi': 'POXRSA', 'permits': 'PORT941BPPRIVSA', 'pop': 'PORPOP'},
    'San Diego': {'hpi': 'SDXRSA', 'permits': 'SAND706BPPRIVSA', 'pop': 'SDIPOP'},
    'San Francisco': {'hpi': 'SFXRSA', 'permits': 'SANF806BPPRIVSA', 'pop': 'SFCPOP'},
    'Seattle': {'hpi': 'SEXRSA', 'permits': 'SEAT653BPPRIVSA', 'pop': 'STWPOP'},
    'Tampa': {'hpi': 'TPXRSA', 'permits': 'TAMP312BPPRIVSA', 'pop': 'TMAPOP'},
    'Washington': {'hpi': 'WDXRSA', 'permits': 'WASH911BPPRIVSA', 'pop': 'WSHPOP'}
}

# Note: Some Case-Shiller indices start later than 1987.
# Data availability for Permits and Population varies by MSA.

In [6]:
# --- Data Fetching Helper ---
def fetch_fred_series(ticker, start, end, freq=None):
    try:
        series = fred.get_series(ticker, observation_start=start, observation_end=end)
        if freq:
            series = series.resample(freq).ffill()
            # Ensure full range coverage
            full_range = pd.date_range(start=start, end=end, freq=freq)
            series = series.reindex(full_range, method='ffill')
        return series
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None


In [7]:

print("Starting Data Download...")
var_data = {}
visuals_data = {}

# 1. Process National Data
print("Processing National Data...")
for name, ticker in national_tickers.items():
    # Determine category and date range
    if name in ['permits', 'population', 'housing starts', 'recessions']:
        # Visuals Category
        start = HOUSING_STARTS_START if name in ['housing starts', 'recessions'] else VISUALS_START_DATE
        series = fetch_fred_series(ticker, start, VISUALS_END_DATE)
        if series is not None:
            visuals_data[f"National_{name}"] = series
            print(f"  Fetched {name} (Visuals)")
    else:
        # VAR Category (cs, cs10, pce, ffr, unemp)
        series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
        if series is not None:
            var_data[name] = series
            print(f"  Fetched {name} (VAR)")

# 2. Process Metropolitan Data
print("\nProcessing Metropolitan Data...")
for city, tickers in metropol_tickers.items():
    for var_name, ticker in tickers.items():
        col_name = city.lower() if var_name == 'hpi' else f"{city}_{var_name}"
        
        if var_name in ['permits', 'pop']:
            # Visuals Category (Permits and Population)
            # Population needs resampling to monthly
            freq = 'MS' if var_name == 'pop' else None
            series = fetch_fred_series(ticker, VISUALS_START_DATE, VISUALS_END_DATE, freq=freq)
            if series is not None:
                visuals_data[col_name] = series
                print(f"  Fetched {col_name} (Visuals)")
        else:
            # VAR Category (HPI)
            series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
            if series is not None:
                var_data[col_name] = series
                print(f"  Fetched {col_name} (VAR)")


Starting Data Download...
Processing National Data...
  Fetched cs (VAR)
  Fetched cs (VAR)
  Fetched cs10 (VAR)
  Fetched cs10 (VAR)
  Fetched pce (VAR)
  Fetched pce (VAR)
  Fetched ffr (VAR)
  Fetched ffr (VAR)
  Fetched unemp (VAR)
  Fetched unemp (VAR)
  Fetched pce_rate (VAR)
  Fetched pce_rate (VAR)
  Fetched permits (Visuals)
  Fetched permits (Visuals)
  Fetched population (Visuals)
  Fetched population (Visuals)
  Fetched housing starts (Visuals)
  Fetched housing starts (Visuals)
  Fetched recessions (Visuals)

Processing Metropolitan Data...
  Fetched recessions (Visuals)

Processing Metropolitan Data...
  Fetched atlanta (VAR)
  Fetched atlanta (VAR)
  Fetched Atlanta_permits (Visuals)
  Fetched Atlanta_permits (Visuals)
  Fetched Atlanta_pop (Visuals)
  Fetched Atlanta_pop (Visuals)
  Fetched boston (VAR)
  Fetched boston (VAR)
  Fetched Boston_permits (Visuals)
  Fetched Boston_permits (Visuals)
  Fetched Boston_pop (Visuals)
  Fetched Boston_pop (Visuals)
  Fetched char

In [8]:

# --- Save Data ---
df_var = pd.DataFrame(var_data)
df_visuals = pd.DataFrame(visuals_data)

# Adjust units if necessary
if 'National_permits' in df_visuals.columns:
    df_visuals['National_permits'] *= 1000


In [9]:

# Save to CSV
var_csv_path = os.path.join(output_dir, 'var_data.csv')
visuals_csv_path = os.path.join(output_dir, 'visual_data.csv')

df_var.to_csv(var_csv_path)
df_visuals.to_csv(visuals_csv_path)

print(f"\nSaved VAR data to {var_csv_path}")
print(f"Saved Visuals data to {visuals_csv_path}")

display(df_var.head())
display(df_visuals.head())


Saved VAR data to data_csv\var_data.csv
Saved Visuals data to data_csv\visual_data.csv


,cs,cs10,pce,ffr,unemp,pce_rate,atlanta,boston,charlotte,chicago,...,miami,minneapolis,new york,phoenix,portland,san diego,san francisco,seattle,tampa,washington
1987-01-01,63.733,63.103814,51.894,6.43,6.6,3.16,NaN,70.266973,63.559024,54.206479,...,68.547905,NaN,74.678440,NaN,41.304891,54.794112,46.955792,NaN,77.441938,64.410215
1987-02-01,64.132,63.804972,52.076,6.10,6.6,3.22,NaN,70.640678,64.028844,55.227274,...,68.919119,NaN,76.043687,NaN,41.444215,55.215119,47.302675,NaN,78.100046,65.132331
1987-03-01,64.468,64.425674,52.214,6.13,6.6,3.41,NaN,71.046774,64.105242,54.811034,...,69.388052,NaN,77.237075,NaN,41.415249,55.487803,47.840213,NaN,77.687495,65.997853
1987-04-01,64.972,65.063393,52.409,6.37,6.3,3.57,NaN,71.411571,64.892990,54.801947,...,69.463461,NaN,78.281527,NaN,41.273786,56.049778,47.984058,NaN,77.819978,66.638734
1987-05-01,65.547,65.815103,52.542,6.85,6.3,3.66,NaN,71.882711,65.222790,55.477466,...,69.746282,NaN,79.675331,NaN,41.395046,56.462099,48.305065,NaN,77.965267,67.444726


,National_permits,National_population,National_housing starts,National_recessions,Atlanta_permits,Atlanta_pop,Boston_permits,Boston_pop,Charlotte_permits,Charlotte_pop,...,San Diego_permits,San Diego_pop,San Francisco_permits,San Francisco_pop,Seattle_permits,Seattle_pop,Tampa_permits,Tampa_pop,Washington_permits,Washington_pop
1959-01-01,NaN,NaN,1657.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-02-01,NaN,NaN,1667.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-03-01,NaN,NaN,1620.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-04-01,NaN,NaN,1590.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-05-01,NaN,NaN,1498.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
